# Ponto de Controle
Este notebook valida e escreve dados transformados no Google Sheets.

In [ ]:
import os
import pandas as pd
import datetime as dt
from extract import read_df
from treat.utils.datas import normalize_date_to_str_DD_M_YYYY
from treat.utils.write_dataframe_to_sheet import write_dataframe_to_sheet
from treat.utils.normalize import normalize_vehicle
from treat.utils.datas import concat_period
from treat.utils.campos_calculados import make_id_ponto_de_controle
from treat.utils.campos_calculados import add_key_creative
from treat.utils.campos_calculados import dedupe_by_key_creative
import numpy as np

In [ ]:
# Flags de execução
"""
Célula  – Imports & parâmetros globais

Define:
- Módulos padrão e helpers do projeto
- Flags de execução e IDs de planilhas via env var
- Constantes de aba, cabeçalho e filtro de data
- Lista de colunas de destino
"""
DRY_RUN = True

# IDs das planilhas via variáveis de ambiente
os.environ["ORIGIN_SHEET_ID"] = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
ORIGIN_SHEET_ID = os.getenv("ORIGIN_SHEET_ID")
os.environ["DEST_SHEET_ID"] = "1DpH5tu4KJKqbA6ueFtf1s1FueBkR4-EtPf5xHyXx8zw"
DEST_SHEET_ID   = os.getenv("DEST_SHEET_ID")

# Garantia de que foram definidas
assert ORIGIN_SHEET_ID is not None and DEST_SHEET_ID is not None, \
    "Defina as variáveis de ambiente ORIGIN_SHEET_ID e DEST_SHEET_ID"


In [ ]:
# Constantes de aba & cabeçalho: nomes centralizados em um só lugar
ORIGIN_TAB    = "modeloGeral"
DEST_TAB      = "IMPULSIONAMENTOS 2025"
HEAD_ROW_DEST = 4  # zero-based (header na linha 5)



In [ ]:
# Filtro temporal & Data mínima – usado no filtro posterior
MIN_DATE = dt.date(2025, 6, 1)


In [ ]:
DEST_COLUMNS = [
    "Periodo",                         # ⬅️ substitui “Data”
    "Campanha",
    "Veiculo",
    "Link conteúdos impulsionados",
    "Agência",
    "Editoria",
    "Objetivo",
    "Meta",
    "Status",
    "Resultado",
    "Criativo",                       # ⬅️ nova coluna
]
assert len(DEST_COLUMNS) == 11, f"DEST_COLUMNS deve ter 11 colunas, mas tem {len(DEST_COLUMNS)}"
print("▶ DEST_COLUMNS definido:", DEST_COLUMNS)
    

In [ ]:
# Leitura da aba de origem – deve executar sem exceção se ORIGIN_SHEET_ID estiver definido
"""
Célula 2 – Leitura + filtro temporal da aba modeloGeral
- Lê df_origin com read_df()
- Converte coluna date para date_dt
- Filtra linhas >= MIN_DATE
- Garante colunas críticas e prepara df_origin
"""
df_origin = read_df(
    sheet_id=ORIGIN_SHEET_ID,
    tab=ORIGIN_TAB,
    header_row=0,
)

# Prioridade: utm_content → ad_name → ad_group_name → ID_Campanha
conds = [
    df_origin["utm_content"].astype(str).str.strip().ne(""), 
    df_origin["ad_name"].astype(str).str.strip().ne(""), 
    df_origin["ad_group_name"].astype(str).str.strip().ne(""), 
    df_origin["ID_Campanha"].astype(str).str.strip().ne(""), 
]
choices = [
    df_origin["utm_content"],
    df_origin["ad_name"],
    df_origin["ad_group_name"],
    df_origin["ID_Campanha"],
]
df_origin["key_creative"] = np.select(conds, choices, default="")

assert (df_origin["key_creative"] != "").all(), \
    "5.1 Falha: há key_creative vazias"

print("▶ 5.1 key_creative gerada com sucesso")
display(df_origin[["utm_content","ad_name","ad_group_name","ID_Campanha","key_creative"]].head())

# ▶ 5.2 • Deduplicar df_origin mantendo primeira ocorrência
orig_len = len(df_origin)
df_origin = df_origin.drop_duplicates("key_creative", keep="first").reset_index(drop=True)
assert len(df_origin) <= orig_len, \
    f"5.2 Falha: tamanho pós-dedup ({len(df_origin)}) > original ({orig_len})"

print(f"▶ 5.2 Deduplicados: {orig_len - len(df_origin)} linhas removidas")




In [ ]:
# Sanitizar tipos de data – converte coluna 'date' para datetime.date e força dtype str para preservar zeros
df_origin = add_key_creative(df_origin)
# 1 · já temos df_origin com coluna key_creative
df_origin_raw = df_origin.copy()
# 2 · aplica deduplicação
df_origin = dedupe_by_key_creative(df_origin)
df_origin['date'] = df_origin['date'].astype(str)
df_origin['date_dt'] = pd.to_datetime(df_origin['date'], errors='coerce').dt.date

# Preview das datas para validação
if DRY_RUN:
    display(df_origin[['date', 'date_dt']].head())
    invalidados = df_origin['date_dt'].isna().sum()
    print(f"Valores inválidos ou não parseados: {invalidados}")


In [ ]:
# Quick-preview em DRY_RUN – exibe head e contagem somente em Dry Run
if DRY_RUN:
    display(df_origin.head())
    print(f"Total de linhas em df_origin: {len(df_origin)}")


In [ ]:
# Assert de colunas críticas – garante que df_origin tenha todas as colunas necessárias
required_columns = ["date", "Campanha", "Veiculo", "URL_do_Anuncio", "objective"]
missing_cols = [col for col in required_columns if col not in df_origin.columns]
if missing_cols:
    raise RuntimeError(f"Colunas críticas ausentes em df_origin: {missing_cols}")


In [ ]:
# Clean-up da coluna auxiliar – remover 'date_dt' apenas após aplicar o filtro de data mínima (útil para debug)
if not DRY_RUN:
    df_origin.drop(columns=["date_dt"], inplace=True)


In [ ]:
# Clonar DataFrame para não mutar a leitura crua
df = df_origin.copy()
df["Criativo"] = df_origin["key_creative"]
assert "Criativo" in df.columns and df["Criativo"].equals(df_origin["key_creative"]), \
    "5.3 Falha: coluna 'Criativo' não corresponde a key_creative"
print("▶ 5.3 Criativo mapeado com sucesso")


In [ ]:
# Normalizar coluna Data e dropar coluna date após criar Data para evitar conflito de nomes
df["Data"] = df["date"].apply(normalize_date_to_str_DD_M_YYYY)
df.drop(columns=["date"], inplace=True)

# Debug: mostrar os primeiros valores de 'Data' e conferir dtype
print("Preview 'Data':")
print(df["Data"].head())
print(f"Tipo de coluna Data: {df['Data'].dtype}, total de linhas: {len(df)}")


In [ ]:
# Normalizar Veiculo – aplicação de normalize_vehicle
df["Veiculo"] = df["Veiculo"].apply(normalize_vehicle)
df.drop(columns=["Veiculo"], inplace=True)              # opcional: remove a antiga


In [ ]:
# Gerar coluna Periodo com concat_period e dropar colunas auxiliares
df["Periodo"] = df.apply(lambda r: concat_period(r.get("start"), r.get("end")), axis=1)
df.drop(columns=["start", "end"], inplace=True)

# Debug: conferir se a geração de 'Periodo' funcionou
print("Preview 'Periodo':")
print(df["Periodo"].head().tolist())
non_empty = df["Periodo"].astype(bool).sum()
print(f"Linhas com Periodo não vazio: {non_empty} de {len(df)}")


In [ ]:
# Mapear colunas diretas
df["Campanha"] = df["Campanha"]
df["Link conteúdos impulsionados"] = df["URL_do_Anuncio"]
df["Objetivo"] = df["objective"]

# Debug prints para verificar mapeamento
print("Preview 'Campanha':", df["Campanha"].head().tolist())
print("Preview 'Link conteúdos impulsionados':", df["Link conteúdos impulsionados"].head().tolist())
print("Preview 'Objetivo",
      df["Objetivo"].head().tolist())

# Asserts para garantir a presença das colunas
assert "Campanha" in df.columns, "Coluna 'Campanha' não encontrada"
assert "Link conteúdos impulsionados" in df.columns, "Coluna 'Link conteúdos impulsionados' não encontrada"
assert "Objetivo" in df.columns, \
       "Coluna 'Objetivo' não encontrada"


In [ ]:
# Colunas constantes / vazias
df["Agência"] = "De Brito"
df["Editoria"] = df["Campanha"]
df["Meta"] = ""
df["Status"] = ""
df["Resultado"] = ""

# Debug prints para verificar preenchimento
print("Preview 'Agência':", df["Agência"].head().tolist())
print("Preview 'Editoria':", df["Editoria"].head().tolist())
print("Valores únicos em 'Agência':", df["Agência"].unique())
print("Contagem não vazia em 'Meta (número)':",
      df["Meta"].astype(bool).sum())
print("Contagem não vazia em 'Status':",
      df["Status"].astype(bool).sum())
print("Contagem não vazia em 'Resultado':",
      df["Resultado"].astype(bool).sum())

# Asserts para garantir colunas e conteúdo esperado
assert "Agência" in df.columns and df["Agência"].eq("De Brito").all(), \
    "Erro em 'Agência': valores diferentes de 'De Brito' ou coluna ausente"
assert "Editoria" in df.columns, "Coluna 'Editoria' ausente"
assert all(df["Meta"] == ""), \
    "'Meta"
assert all(df["Status"] == ""), "'Status' deve ser completamente vazio"
assert all(df["Resultado"] == ""), "'Resultado' deve ser completamente vazio"


In [ ]:
# Debug prints e asserts para verificar ordem e conteúdo das colunas


# Reordenar / reindexar com DEST_COLUMNS e preencher vazios
df_transf = df.reindex(columns=DEST_COLUMNS, fill_value="")
print("Colunas em df_transf:", df_transf.columns.tolist())
assert set(DEST_COLUMNS).issubset(df_transf.columns), (
    f"Colunas fora de ordem ou faltando: {df_transf.columns.tolist()}"
)

# Mostrar as primeiras linhas para confirmação visual
display(df_transf.head())
print(f"Total de linhas em df_transf: {len(df_transf)}")


In [ ]:
# ▶ Gerar __ID__ após garantir colunas necessárias
print("▶ Gerando __ID__ em df_transf...")
df_transf["__ID__"] = df_transf.apply(make_id_ponto_de_controle, axis=1)

# ▶ Validações
assert df_transf["__ID__"].isna().sum() == 0, "❌ Há NaNs em __ID__ em df_transf — verifique campos obrigatórios vazios"
assert df_transf["__ID__"].nunique() == len(df_transf), f"❌ __ID__ duplicado em df_transf ({df_transf['__ID__'].nunique()} únicos vs {len(df_transf)} linhas)"
print("✅ __ID__ gerado com sucesso em df_transf")


In [ ]:
# 3.7.1 – Dropar colunas auxiliares 'start' e 'end' se ainda existirem
aux_cols = [c for c in ["start", "end"] if c in df_transf.columns]
if aux_cols:
    df_transf.drop(columns=aux_cols, inplace=True)

# Debug: confirmar que as colunas auxiliares foram removidas
print("Colunas após remoção de 'start' e 'end':", df_transf.columns.tolist())


In [ ]:
# ---------------------------------------------------------------------------
# Mapear coluna 'Criativo' no DataFrame transformado (df_transf)
# ---------------------------------------------------------------------------

"""
Insere em df_transf a coluna 'Criativo', herdando de df_origin['key_creative'].
Critérios de aceite:
  - A coluna 'Criativo' deve existir em df_transf.
  - Não deve haver valores vazios ou NaN em 'Criativo'.
  - A ordem dos valores deve corresponder à de df_transf.
"""

# 1 · Garantir alinhamento entre df_origin e df_transf
assert len(df_transf) == len(df_origin), (
    f"Linhas incompatíveis: df_transf tem {len(df_transf)}, "
    f"mas df_origin tem {len(df_origin)}"
)

# 2 · Mapear a nova coluna
df_transf = df_transf.copy()
df_transf["Criativo"] = df_origin["key_creative"].values

# 3 · Debug / validação
print("Preview da coluna 'Criativo':", df_transf["Criativo"].head().tolist())
assert "Criativo" in df_transf.columns, "❌ Coluna 'Criativo' não foi adicionada."
assert not df_transf["Criativo"].isna().any(), (
    "❌ Há valores NaN em 'Criativo' – verifique a geração de key_creative."
)
print(f"✅ Total de valores únicos em 'Criativo': {df_transf['Criativo'].nunique()}")


In [ ]:
# ---------------------------------------------------------------------------
# Consolidar aba de destino “IMPULSIONAMENTOS 2025” preservando dropdowns,
# check-boxes e demais valores FORMATADOS, e gerar __ID__ final.
# ---------------------------------------------------------------------------
import os, re, unicodedata
import pandas as pd
from treat.utils.get_google_client import get_google_client
from treat.utils.campos_calculados import make_id_ponto_de_controle

# ————— constantes já definidas no notebook ——————————————
# DEST_SHEET_ID, DEST_TAB, HEAD_ROW_DEST (5), DEST_COLUMNS (lista de 11 colunas, incluindo "Veiculo")

HEADER_ROW_SHEET = HEAD_ROW_DEST + 1

# 1 • Conectar ao worksheet via gspread
CREDS_PATH = os.getenv("GOOGLE_CREDS_PATH", "creds.json")
gclient    = get_google_client(CREDS_PATH)
ws_dest    = gclient.open_by_key(DEST_SHEET_ID).worksheet(DEST_TAB)

# 2 • Captura cabeçalho “bruto” (linha 5 → índice 5 no Sheets)
header_raw = ws_dest.row_values(HEADER_ROW_SHEET)
print("Header bruto:", header_raw)

# 3 • Limpa cada label de cabeçalho
def _clean(label: str) -> str:
    txt = unicodedata.normalize("NFKD", label or "")
    txt = "".join(c for c in txt if not unicodedata.combining(c))
    txt = txt.replace("\n", " ").strip()
    return re.sub(r"\s{2,}", " ", txt)

cleaned = [_clean(c) for c in header_raw]
print("Header limpo :", cleaned)

# 4 • Puxa linhas formatadas a partir da linha 6 (HEAD_ROW_DEST+1)
body = ws_dest.get_values(
    f"A{HEADER_ROW_SHEET+1}:K",

)
assert body, "❌ body veio vazio: não trouxe nenhuma linha de dados"

# 5 • Normaliza cada linha para ter exatamente 11 colunas
n_cols = len(DEST_COLUMNS)
normalized = [(row + [""]*n_cols)[:n_cols] for row in body]

# 6 • Monta DataFrame com o cabeçalho oficial
df_dest = pd.DataFrame(normalized, columns=DEST_COLUMNS)
if "Criativo" not in df_dest.columns:
    df_dest["Criativo"] = ""
assert set(DEST_COLUMNS).issubset(df_dest.columns)

print("Colunas após montagem:", df_dest.columns.tolist())



# 8 • Limpa linhas totalmente vazias (bordas/formatação)
df_dest = (
    df_dest
    .replace("", pd.NA)
    .dropna(how="all")
    .reset_index(drop=True)
)
print(f"Linhas válidas em df_dest após dropna: {len(df_dest)}")

# 9 • Gera coluna __ID__ — somente aqui, com todos os campos (incluindo key_creative já embutido)
df_dest["__ID__"] = df_dest.apply(make_id_ponto_de_controle, axis=1)

# 9.1 • Deduplica mantendo o 1º registro de cada ID
before = len(df_dest)
df_dest = df_dest.drop_duplicates("__ID__", keep="first").reset_index(drop=True)
removed = before - len(df_dest)
print(f"🧹 Removidas {removed} linhas duplicadas de __ID__ (mantida a 1ª ocorrência)")

# 10 • Validações finais (já com df_dest deduplicado)
assert df_dest["__ID__"].notna().all(), "❌ Há NaNs em __ID__"
assert df_dest["__ID__"].nunique() == len(df_dest), \
    f"❌ __ID__ duplicados ainda presentes: {df_dest['__ID__'].nunique()} únicos vs {len(df_dest)} linhas"

print("▶ __ID__ gerado, deduplicado e validado com sucesso")


# 10 • Validações finais
assert df_dest["__ID__"].notna().all(), "❌ Há NaNs em __ID__"
assert df_dest["__ID__"].nunique() == len(df_dest), \
    f"❌ __ID__ duplicados: {df_dest['__ID__'].nunique()} únicos vs {len(df_dest)} linhas"
print("▶ __ID__ gerado e validado com sucesso")

# 11 • Pré-visualização
display(df_dest.head())
print(df_dest.apply(lambda c: (c != "") & (c.notna())).sum())




In [ ]:
# ---------------------------------------------------------------------------
# Gerar coluna de identificador único (__ID__) em df_transf
# Mesmo make_id_ponto_de_controle usado para df_dest
# ---------------------------------------------------------------------------

# 1 · Cria/atualiza a coluna __ID__ em df_transf
df_transf["__ID__"] = df_transf.apply(make_id_ponto_de_controle, axis=1)

# 2 · Validação rápida
print(f"__ID__ gerados em df_transf: {df_transf['__ID__'].nunique()} (únicos) / {len(df_transf)} (linhas)")
assert df_transf["__ID__"].isna().sum() == 0, "Há valores NaN em __ID__ em df_transf — verifique campos vazios"

# 3 · Pré-visualização
display(df_transf.head()[["Periodo", "Campanha", "Veiculo", "__ID__"]])


In [ ]:
# ---------------------------------------------------------------------------
# Deduplicação: filtra apenas as linhas novas em df_transf
# ---------------------------------------------------------------------------

# 1 • Cria o DataFrame somente com os registros cujo __ID__ não está em df_dest
novos = df_transf[~df_transf["__ID__"].isin(df_dest["__ID__"])].copy()

# 2 • Validação rápida: nunca adicionar mais registros do que existem na origem
assert len(novos) <= len(df_transf), (
    f"Erro de deduplicação: len(novos)={len(novos)} maior que len(df_transf)={len(df_transf)}"
)

# 3 • Exibe quantas linhas novas foram identificadas
print(f"Linhas novas após deduplicação: {len(novos)}")

# 4 • Pré-visualização das primeiras entradas novas
display(novos.head())
